# Inspect PartEdit-Bench For Part-Level TDM Localization

This notebook checks whether PartEdit-Bench can support Harry Yang's requested pilot study: 10-15 cases balanced by target part size, with ground-truth masks for evaluating Follow-Your-Shape TDM localization.

## Goal

Before renting a GPU or running Follow-Your-Shape, confirm the dataset fields, image/mask formats, prompt structure, and mask-area distribution. The notebook should produce a small candidate table, not download or commit the full dataset into the repository.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import get_dataset_config_names, load_dataset
from PIL import Image

REPO_ROOT = Path.cwd()
DATASET_ID = "Aleksandar/PartEdit-Bench"
OUTPUT_DIR = REPO_ROOT / "core" / "data" / "partedit_subset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREFERRED_SPLIT = "real"
IMAGE_FIELD = "original_image"
EDITED_REFERENCE_FIELD = "partedit"
MASK_FIELD = "gt_mask"
SOURCE_PROMPT_FIELD = "prompt_original"
TARGET_PROMPT_FIELD = "prompt_changed"
PART_FIELD = "part"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

## 1. Inspect Dataset Configs And Splits

In [ ]:
configs = get_dataset_config_names(DATASET_ID)
configs

In [ ]:
dataset_kwargs = {}
if configs:
    dataset_kwargs["name"] = configs[0]

dataset = load_dataset(DATASET_ID, **dataset_kwargs)
dataset

## 2. Inspect Actual Schema And Example Content

In [ ]:
split_name = PREFERRED_SPLIT if PREFERRED_SPLIT in dataset else list(dataset.keys())[0]
split = dataset[split_name]

print("split:", split_name)
print("num rows:", len(split))
split.features

In [ ]:
sample = split[0]

field_rows = []
for field_name, value in sample.items():
    if isinstance(value, Image.Image):
        preview = f"PIL.Image size={value.size} mode={value.mode}"
    else:
        preview = value
    field_rows.append({
        "field": field_name,
        "python_type": type(value).__name__,
        "preview": preview,
    })

pd.DataFrame(field_rows)

In [ ]:
task_fields = [
    "id",
    "class_name",
    "subject",
    "part",
    "edit",
    "seed",
    SOURCE_PROMPT_FIELD,
    TARGET_PROMPT_FIELD,
    "p2p_prompt",
    "p2p_template",
    "instructp2p_edit1",
    "instructp2p_edit2",
    "instructp2p_edit3",
]

pd.DataFrame([
    {"field": field, "value": sample[field]}
    for field in task_fields
    if field in sample
])

## 3. Set Follow-Your-Shape Field Mapping

PartEdit-Bench already provides the fields needed for this diagnostic. `prompt_original` and `prompt_changed` map directly to Follow-Your-Shape's source and target prompts, while `gt_mask` is used only for evaluation.

In [ ]:
field_mapping = pd.DataFrame([
    {"experiment_role": "source image", "dataset_field": IMAGE_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "source prompt", "dataset_field": SOURCE_PROMPT_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "target prompt", "dataset_field": TARGET_PROMPT_FIELD, "used_for": "Follow-Your-Shape input"},
    {"experiment_role": "ground-truth part mask", "dataset_field": MASK_FIELD, "used_for": "localization evaluation only"},
    {"experiment_role": "target part label", "dataset_field": PART_FIELD, "used_for": "case description and grouping"},
    {"experiment_role": "PartEdit reference image", "dataset_field": EDITED_REFERENCE_FIELD, "used_for": "optional qualitative reference"},
])

field_mapping

In [ ]:
required_fields = [
    IMAGE_FIELD,
    MASK_FIELD,
    SOURCE_PROMPT_FIELD,
    TARGET_PROMPT_FIELD,
    PART_FIELD,
]
missing_fields = [field for field in required_fields if field not in sample]

if missing_fields:
    raise KeyError(f"Missing required fields: {missing_fields}. Available fields: {list(sample.keys())}")

print("Required PartEdit-Bench fields are available.")

## 4. Visual Check One Example

In [ ]:
def as_mask_array(value):
    if isinstance(value, Image.Image):
        arr = np.asarray(value.convert("L"))
    else:
        arr = np.asarray(value)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr > 0


def mask_area_ratio(mask_value):
    mask = as_mask_array(mask_value)
    return float(mask.mean())


image = sample[IMAGE_FIELD]
mask = as_mask_array(sample[MASK_FIELD])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("original_image")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title(f"gt_mask ratio={mask.mean():.3f}")
axes[2].imshow(image)
axes[2].imshow(mask, alpha=0.35, cmap="Reds")
axes[2].set_title("mask overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 5. Compute Mask-Area Distribution

In [ ]:
records = []
for idx in range(len(split)):
    row = split[idx]
    records.append({
        "dataset_split": split_name,
        "dataset_index": idx,
        "id": row["id"],
        "class_name": row["class_name"],
        "subject": row["subject"],
        "part": row[PART_FIELD],
        "edit": row["edit"],
        "prompt_original": row[SOURCE_PROMPT_FIELD],
        "prompt_changed": row[TARGET_PROMPT_FIELD],
        "mask_area_ratio": mask_area_ratio(row[MASK_FIELD]),
    })

mask_table = pd.DataFrame(records).sort_values("mask_area_ratio")
mask_table

In [ ]:
ax = mask_table["mask_area_ratio"].hist(bins=30, figsize=(8, 4))
ax.set_title("PartEdit-Bench target mask area ratios")
ax.set_xlabel("mask area / image area")
ax.set_ylabel("case count")

## 6. Build A Candidate Balanced Subset

This produces a first-pass candidate table. Review examples visually before turning it into the final `cases.json` manifest.

In [ ]:
def size_bucket(mask_ratio):
    if mask_ratio < 0.05:
        return "small"
    if mask_ratio < 0.15:
        return "medium"
    return "large"


mask_table["part_size"] = mask_table["mask_area_ratio"].map(size_bucket)

candidate_subset = (
    mask_table.groupby("part_size", group_keys=False)
    .apply(lambda frame: frame.sort_values("mask_area_ratio").head(5))
    .reset_index(drop=True)
)

candidate_subset[[
    "dataset_split",
    "dataset_index",
    "id",
    "part_size",
    "mask_area_ratio",
    "class_name",
    "subject",
    "part",
    "edit",
    "prompt_original",
    "prompt_changed",
]]

In [ ]:
preview_path = OUTPUT_DIR / "cases_preview.csv"
candidate_subset.to_csv(preview_path, index=False)
preview_path

## Next Checks

- Confirm the guessed field names match the actual dataset schema.
- Visually inspect candidate source images and masks.
- Replace the preview CSV with a reviewed JSON manifest for the 10-15 pilot cases.
- Do not commit downloaded images or masks.